# 第 9 章 · 学习与适应（OpenEvolve）

用进化算法改进一段代码：LLM 改写 → 自动评估打分 → 保留最好的 → 再改写。改进的是**代码本身**。

书里只给了 6 行构造代码，三个路径都是占位符，且 API 与 0.4.1 不一致（书传 `config_path=`，实际要传 `Config` 对象）。

下面用 `../../ch09_adaptation/` 里的三件套跑一次真实进化：初始程序是三重循环的最大子段和，评估器同时看正确性与速度。

kernel 选 `Python (agentic-evolve)`（独立 venv，未装 langchain / ADK）。


In [1]:
import json, os, sys
from pathlib import Path

# 【适配】kernel 的工作目录不一定是 notebook 所在目录（VS Code 常以工作区根启动），
#   所以向上找「含 ch09_adaptation/」的那层，而不是写死 ../..
def _find_dir(name="ch09_adaptation"):
    starts = [Path.cwd()]
    ipynb = globals().get("__vsc_ipynb_file__")      # VS Code 会注入 notebook 路径
    if ipynb:
        starts.insert(0, Path(ipynb).parent)
    for start in starts:
        for d in (start, *start.parents):
            cand = d / name
            if (cand / "evaluator.py").exists():
                return cand
    raise FileNotFoundError(f"找不到 {name}/（当前工作目录：{Path.cwd()}）")

DIR = _find_dir()
sys.path.insert(0, str(DIR))

print("三件套目录      :", DIR)
print("BOOK_API_KEY 已由 venv 启动引导载入:", bool(os.getenv("BOOK_API_KEY")))

# 先量一次基线，后面好对比
import evaluator
base = evaluator.evaluate(str(DIR / "initial_program.py"))
print("\n初始程序指标：")
for k, v in base.items():
    print(f"  {k:16} {v}")


三件套目录      : /home/stvli/agentic_design_patterns/ch09_adaptation
BOOK_API_KEY 已由 venv 启动引导载入: True

初始程序指标：
  correctness      1.0
  speed            0.07872197711228608
  runtime_seconds  0.02540586597751826
  combined_score   0.7236165931336858


In [2]:
# 【适配】书里直接 OpenEvolve(config_path=...)；0.4.1 要先 load_config 得到 Config 对象
from openevolve import OpenEvolve
from openevolve.config import load_config

config = load_config(str(DIR / "configs" / "deepseek-flash.yaml"))
config.log_level = "WARNING"      # 演示时别刷 INFO 日志
# 也可换成任一本地权重（需先启动对应 shim，见 ch09_adaptation/README.md）：
# config = load_config(str(DIR / "configs" / "qwen3.5-0.8b.yaml"))
# config = load_config(str(DIR / "configs" / "qwen3.5-2b.yaml"))
# config = load_config(str(DIR / "configs" / "qwen3.5-4b.yaml"))
# config = load_config(str(DIR / "configs" / "qwen3.5-9b.yaml"))
config.max_iterations = 2          # 演示只要一轮有效改写，不必跑满 1000

evolve = OpenEvolve(
    initial_program_path=str(DIR / "initial_program.py"),
    evaluation_file=str(DIR / "evaluator.py"),
    config=config,
    output_dir=str(DIR / "runs" / "_notebook"),
)

best = await evolve.run(iterations=config.max_iterations)      # ★ 书里那行 evolve.run(iterations=1000)
print("\n最佳程序指标：")
for name, value in best.metrics.items():
    print(f"  {name:16} {value}")



最佳程序指标：
  correctness      1.0
  speed            1.0
  runtime_seconds  3.0521012376993895e-05
  combined_score   1.0


In [3]:
# 进化出来的代码长什么样
best_file = DIR / "runs" / "_notebook" / "best" / "best_program.py"
print(best_file.read_text())

# 与基线的对比
after = evaluator.evaluate(str(best_file))
print("初始  :", {k: round(v, 6) if isinstance(v, float) else v for k, v in base.items()})
print("进化后:", {k: round(v, 6) if isinstance(v, float) else v for k, v in after.items()})
print(f"\n速度提升约 {base['runtime_seconds'] / after['runtime_seconds']:.0f} 倍")


"""最大子段和：初始实现是三重循环的暴力版本 —— 正确，但很慢。

OpenEvolve 只允许改写 EVOLVE-BLOCK 之间的代码。
"""

# EVOLVE-BLOCK-START
def max_subarray_sum(arr):
    """返回 arr 中连续子数组的最大和（arr 至少一个元素）。"""
    best = current = arr[0]
    for value in arr[1:]:
        current = max(value, current + value)
        if current > best:
            best = current
    return best
# EVOLVE-BLOCK-END

初始  : {'correctness': 1.0, 'speed': 0.078722, 'runtime_seconds': 0.025406, 'combined_score': 0.723617}
进化后: {'correctness': 1.0, 'speed': 1.0, 'runtime_seconds': 2.2e-05, 'combined_score': 1.0}

速度提升约 1137 倍
